# Integridad · Salas y borrados

[Inicio](../../tests.ipynb) · Anterior: [Integridad · Identidad y partidas](identidad_y_partidas.ipynb) · Siguiente: [Integridad · Ciclo, métricas, sesión y textos](ciclo_metricas_sesion_y_textos.ipynb)

Las fichas incluyen los criterios del Plan Maestro. Las referencias U/G/F/W/D y las rutas abreviadas en **Evidencia** se interpretan en la [matriz maestra](../../docs/test/qa-master-matrix.md). Una ejecución local no implica un resultado publicado ni prueba un control de entrega.

## RI-05

- **Criterio:** Un jugador tiene una sola participación por partida.
- **Método previsto:** API, WebSocket o SQL, según la capa. Garantía prevista: BD: UNIQUE (partida, usuario).
- **Estado:** **V — Verificado**.
- **Evidencia:** En la matriz (**N**): `supabase/db/init.sql` declara `UNIQUE (game_id, user_id)`; no se identificó una prueba que intente insertar un duplicado en PostgreSQL.
  
  Evidencia local posterior o adicional: PostgreSQL 16/Alembic 0005 rechazó el duplicado con SQLSTATE 23505 y conservó la fila original (Engram #993); misma corrida que RI-03 y RI-04. [Fuente de prueba](../../backend/tests/features/participation.feature).
- **Brecha o límite:** Alcance: la pareja partida/usuario, no dos ejecuciones distintas.

**Gherkin ejecutable (original en inglés)** · [Fuente](../../backend/tests/features/participation.feature).

```gherkin
Feature: Completed matches preserve participation integrity

  @RI-04 @RI-05
  Scenario: A completed race stores exactly one participation per player
    Given a live race with two signed-in members
    When both members complete the race
    Then the game is linked to the live room and has one participation per player
    And PostgreSQL rejects a duplicate game and user pair without adding a row
```

## RI-06

- **Criterio:** Un usuario pertenece a una sola sala a la vez.
- **Método previsto:** API, WebSocket o SQL, según la capa. Garantía prevista: BD + Servidor.
- **Estado:** **V — Verificado**.
- **Evidencia:** En la matriz (**P**): `test_rooms_api.py` (**U**) comprueba pertenencia única en secuencia; `backend/alembic/versions/0003_salas_multijugador.py` declara índice único. Falta intentar ingresos concurrentes a salas distintas en PostgreSQL.
  
  Evidencia local posterior o adicional: Dos solicitudes simultáneas del mismo usuario para salas distintas dejaron una sola membresía en PostgreSQL. Corrida compartida con RI-07 y RI-08: 5 escenarios, 16 pasos, Alembic 0005 (Engram #995–996). [Fuente de prueba](../../backend/tests/features/room_integrity.feature).
- **Brecha o límite:** La barrera coordina solicitudes; no fija el orden de las transacciones.

**Gherkin ejecutable (original en inglés)** · [Fuente](../../backend/tests/features/room_integrity.feature).

```gherkin
Feature: Room integrity on migrated PostgreSQL

  @RI-06
  Scenario: One player requests admission to two rooms concurrently
    Given two waiting rooms with different hosts
    When one new player requests admission to both rooms concurrently
    Then one admission succeeds and PostgreSQL stores only one membership for that player
```

## RI-07

- **Criterio:** El código de sala es único; una sala admite máximo 10 jugadores y no acepta ingresos con una partida en curso.
- **Método previsto:** API, WebSocket o SQL, según la capa. Garantía prevista: BD: UNIQUE + Servidor.
- **Estado:** **V — Verificado**.
- **Evidencia:** En la matriz (**P**): `test_rooms_api.py` (**U**) cubre capacidad en secuencia e ingreso durante partida; `0003_salas_multijugador.py` declara unicidad del código. Falta provocar colisión de código y verificar el límite de diez ante ingresos simultáneos en PostgreSQL.
  
  Evidencia local posterior o adicional: La corrida de RI-06 a RI-08 comprobó colisión de código al actualizar, competencia por la última plaza y rechazo con carrera activa (Engram #995–996). [Fuente de prueba](../../backend/tests/features/room_integrity.feature).
- **Brecha o límite:** La colisión fuerza la restricción en PostgreSQL; no mide la frecuencia de colisiones al generar códigos.

**Gherkin ejecutable (original en inglés)** · [Fuente](../../backend/tests/features/room_integrity.feature).

```gherkin
Feature: Room integrity on migrated PostgreSQL

  @RI-07
  Scenario: PostgreSQL enforces unique room codes
    Given two waiting rooms with different hosts
    Then PostgreSQL rejects changing one room code to the other room's code

  @RI-07
  Scenario: Concurrent admissions compete for the last seat
    Given a waiting room with nine members
    When two new players request admission to the room concurrently
    Then one admission succeeds and the room has exactly ten members

  @RI-07
  Scenario: A new player cannot enter a live race
    Given a live race with two signed-in members
    Then a new player's admission is refused and the live roster is unchanged
```

## RI-08

- **Criterio:** Solo el anfitrión configura, expulsa, inicia y cierra la sala; si la abandona, la sala y sus membresías se eliminan.
- **Método previsto:** API, WebSocket o SQL, según la capa. Garantía prevista: Servidor + BD (CASCADE).
- **Estado:** **V — Verificado**.
- **Evidencia:** En la matriz (**P**): Restricciones de configuración, expulsión e inicio, cierre del anfitrión y salida en `test_rooms_api.py` y `test_ws_rooms.py` (**U**). Faltan rechazo explícito de cierre por un miembro y comprobación de cascada en PostgreSQL.
  
  Evidencia local posterior o adicional: La corrida de RI-06 a RI-08 verificó rechazo de cierre por un miembro, cascada de membresías al borrar la sala y limpieza al salir el anfitrión (Engram #995–996); se complementa con las pruebas de permisos del corte. [Fuente de prueba](../../backend/tests/features/room_integrity.feature).
- **Brecha o límite:** No se vuelve a contar la ejecución compartida.

**Gherkin ejecutable (original en inglés)** · [Fuente](../../backend/tests/features/room_integrity.feature).

```gherkin
Feature: Room integrity on migrated PostgreSQL

  @RI-08
  Scenario: A member cannot close the room and the host can leave
    Given a room with two signed-in members
    When the member requests to close the room
    Then the request is forbidden and the waiting room is unchanged
    And PostgreSQL cascades a room deletion to its memberships
    When the host leaves the room
    Then the room and its memberships are gone
```

## RI-13

- **Criterio:** Al borrar un usuario se borran sus participaciones y salas; el récord de un texto queda vacío; un texto con partidas no se puede borrar.
- **Método previsto:** API, WebSocket o SQL, según la capa. Garantía prevista: BD: CASCADE, SET NULL, RESTRICT.
- **Estado:** **V — Verificado**.
- **Evidencia:** En la matriz (**P**): Solo estático: `supabase/db/init.sql` define `CASCADE`, `SET NULL` y `RESTRICT`. Falta verificar borrados en PostgreSQL migrado y aclarar si «récord vacío» significa quitar el titular o también el tiempo ideal.
  
  Evidencia local posterior o adicional: En 0005, el borrado dejaba el tiempo ideal anterior sin titular. En PostgreSQL 16.15 con Alembic 0006 pasaron 2 escenarios y 8 pasos; se verificaron cascadas, titular nulo y retorno al tiempo base, preservación de otras marcas y rechazo de borrado de un texto referenciado (Engram #997–999). [Fuente de prueba](../../backend/tests/features/deletion_integrity.feature).
- **Brecha o límite:** «Récord vacío» significa quitar el titular y restablecer el tiempo base; el SQL manual de entonces aún terminaba en 0005.

**Gherkin ejecutable (original en inglés)** · [Fuente](../../backend/tests/features/deletion_integrity.feature).

```gherkin
Feature: Referential integrity on deletion

  @RI-13
  Scenario: Deleting a record holder cascades to owned rows and resets their records
    Given a record holder hosts a room and has played a match with another user
    When PostgreSQL deletes the record holder
    Then the holder's participation, hosted room, and its memberships are gone
    And all the holder's texts have no record and use the default ideal time
    And the other user's record and the holderless text are unchanged

  @RI-13
  Scenario: A match prevents deletion of its text
    Given a record holder hosts a room and has played a match with another user
    When PostgreSQL attempts to delete the text within a savepoint
    Then the text and its match retain their original state after rollback
```

## Reproducción local opcional

Prepare el entorno desechable de la [guía de pruebas](../../test/README.md). Ejecute solo la celda que necesita, con el kernel iniciado en la raíz de Typr o dentro de `notebooks/`. No ejecute en paralelo. Abrir el cuaderno no inicia pruebas; la salida describe una corrida local, no cambia el estado de las fichas. Las celdas de backend ejecutan la suite indicada completa, no solo los casos de esta página.

### Criterios de aceptación · RI-05 a RI-08 y RI-13

**Plan (§5.6):** Una participación por jugador y partida, y una sala por usuario. El código de sala es único, el aforo es de 10 y no se admite a nadie durante una carrera. Solo el anfitrión administra la sala; si sale, se eliminan la sala y sus membresías. Borrar un usuario elimina sus participaciones y salas y deja vacío su récord; un texto con partidas no se puede borrar.

**Qué comprueba esta celda:** Ejecuta todo el Gherkin del backend, sin filtro. Los escenarios relevantes comprueban que PostgreSQL rechaza la participación y el código duplicados, que solo prospera un ingreso concurrente a dos salas o al último cupo, y que una carrera en curso no admite otro jugador. También comprueban el rechazo del cierre por un miembro, el borrado en cascada al salir el anfitrión, el restablecimiento de los récords del usuario borrado y el rechazo del borrado de un texto referenciado.

**Límite:** Para los permisos de RI-08, este Gherkin no cubre por sí solo configurar, expulsar e iniciar. RI-05 comparte la comprobación multijugador con RI-04.

In [ ]:
from pathlib import Path
from runpy import run_path

directorio = Path.cwd().resolve()
for raiz in (directorio, *directorio.parents):
    if (raiz / "docker-compose.test.yml").is_file() and (raiz / "notebooks/ejecutar_qa.py").is_file():
        break
else:
    raise RuntimeError("Inicie el kernel desde la raíz de Typr o dentro de notebooks/.")

run_path(str(raiz / "notebooks/ejecutar_qa.py"))["ejecutar"]('backend-gherkin')

[Inicio](../../tests.ipynb) · Anterior: [Integridad · Identidad y partidas](identidad_y_partidas.ipynb) · Siguiente: [Integridad · Ciclo, métricas, sesión y textos](ciclo_metricas_sesion_y_textos.ipynb)